# 2. Model training

The first epochs can train the decoder while freezing the encoder; later epochs adapt both. Epoch selection uses a training-only holdout, followed by optional full-training refit. Dummy presets run one short CPU FP32 epoch. Official R/RS weights are downloaded and SHA-verified on first use; private fine-tuned weights are not distributed. Read the benchmark recipe notes before claiming reproduction.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "waterec_ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository")
sys.path.insert(0, str(ROOT))
from waterec_ocr.waterec_config import load_config
from waterec_ocr.waterec_models import default_checkpoint
CONFIG_PATH = os.environ.get("WATEREC_CONFIG", "configs/dummy_alpr.yaml")
config = load_config(CONFIG_PATH)
checkpoint = default_checkpoint(config)
deployment = config["output_dir"] / "deployment"
examples = sorted((config["dataset_root"] / "val").glob("*.png"))
IMAGE_PATH = examples[0] if examples else config["dataset_root"] / "val/example.png"
def run(*args):
    return subprocess.run([sys.executable, "-m", "waterec_ocr.waterec_cli", *args, "--config", CONFIG_PATH], cwd=ROOT, check=True)
print("Profile:", CONFIG_PATH, "variant:", config["variant"], "patch budget:", config["tokens"])


In [ ]:
RUN_TRAINING = os.environ.get("RUN_TRAINING", "0") == "1"
if RUN_TRAINING:
    run("train")
else:
    print("Set RUN_TRAINING=True in this cell (or the environment) after reviewing the profile.")
print("Expected checkpoint:", checkpoint)


For GPU training use `device: cuda`, a supported precision and a batch size appropriate to your hardware. The price-2048 starting profile uses batch 1 conservatively; the reported research recipe used specialized checkpointing/SDPA at batch 16. Keep a new `output_dir` when changing data or configuration. Unsupported/oversized labels raise errors rather than being truncated.